# Regenerate s1K reasoning and answers with DeepSeek Pro

Run on **Colab CPU**. Add `HF_TOKEN` (read access) and `DEEPSEEK_API_KEY` to Colab Secrets
and enable notebook access. This notebook contains its helper code: no Git push or GPU
is required. Use a fresh runtime when opening a new version of this notebook.

Source: `Seungjun/dp_removed_s1K-1.1`, pinned to the same **996 rows** used by the project
before the SFT length filter. Every original column and row is preserved in the full export.

DeepSeek Pro solves the original **question only**; it is not given the old reasoning,
answer, reference solution, or grade. Thinking is enabled. Its native `reasoning_content`
becomes `deepseek-v4-pro_reasoning`; its final `content` becomes `deepseek-v4-pro_answer`.
The final response must contain a **natural worked reasoning process, without fixed
section headings**. Planning establishes the approach; evaluation checks intermediate
results as they arise; reflection revises a failing approach; exploration considers an
alternative when useful. Instructive errors or dead ends from the API thinking may be
included with their checks and corrections. Errors must not be invented, and a smooth
solution need not contain any. No word or token length target is requested.
Here "answer" means the API's final `content` field, not a literal `<answer>` wrapper.
There is **no answer-correctness filter**.

**The intended new training target is `deepseek-v4-pro_answer`: the complete natural
worked solution.** `deepseek-v4-pro_reasoning` retains raw API thinking for inspection.
This notebook does not change the existing SFT loader to use either new column.

**Updated format:** reopen this notebook in a fresh Colab runtime and rerun the smoke test.
This replaces our previous four-section template, not a published Kimi prompt.
The new prompt/code creates a fresh output folder, leaving previous results intact.
Seed 42 selects the same 20 questions, so you can compare the old and updated smoke runs.

1. Run setup, settings, and source-loading cells.
2. Run **Smoke test** for 20 reproducibly random examples and download the five-column CSV.
3. Use the separate **Full run** cell to generate the entire dataset, reusing completed rows.

A default **Run all** runs the paid smoke test only; `RUN_FULL_GENERATION` starts false.
Both modes save each completed response immediately to Google Drive. Rerun a cell after
interruption to resume; unsuccessful rows are attempted again, completed rows are reused.
Use only **one notebook/runtime at a time** for the same output folder.

In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/LG-AIME-S1-DeepSeek"

In [ ]:
# @title Install CPU dependencies, mount Drive, and read secrets
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "datasets==5.0.1", "huggingface-hub==0.36.2", "httpx==0.28.1",
    "PyYAML==6.0.3", "tqdm==4.67.1"])
from google.colab import drive, userdata, files
drive.mount("/content/drive")
HF_TOKEN = userdata.get("HF_TOKEN")
DEEPSEEK_API_KEY = userdata.get("DEEPSEEK_API_KEY")
if not HF_TOKEN or not DEEPSEEK_API_KEY:
    raise ValueError("Enable notebook access to HF_TOKEN and DEEPSEEK_API_KEY in Secrets")

In [ ]:
# @title Install the bundled, tested project code (self-contained notebook)
import base64, io, sys, zipfile
from pathlib import Path
CODE_ROOT = Path("/content/lg-s1-regeneration")
CODE_ROOT.mkdir(parents=True, exist_ok=True)
BUNDLE = 'UEsDBBQAAAAIAAAAIVwAAAAAAgAAAAAAAAASAAAAY29tbW9uL19faW5pdF9fLnB5AwBQSwMEFAAAAAgAAAAhXHQtxyXcAgAAhggAAAwAAABjb21tb24vaW8ucHnVVUtv2zAMvudXEDrJa2psBbZDgOyybodh6AbscemKQI7pRY0jGZLcJCj630dKTuLEwbDDephOJmk+vk8kJYS4bp0qaoSPXz/fQKVr9KBMCapp0JSX1tRbuLetM6r2sNZhAQ7n9gGTkzYBnWubgCUEpWufCyFGI71qrAuwUH5R62In3ntrdt/WjypnV9CowL9Ap/5C4mg0KrGCUv9CH+SDqlvM4PIt+OAmI6DjMFA9u+i5X6ir129kNPHhNHnZrhqfnMfgKfRsiVs//eZYRuNbhzPl51pPPxAwzHI0c1uizGKYLF/gpisg6+pZOx1wxsElFz2GVFkqiTUwjeVHa7bX5o1yaEK+WpbaySTs69hoH2Z2GcXkEpCJUG5L0aI7Uz7zbVXpTYycp2+4AJGHVSOSW7yYvW9u6e6kWIsMlIdqMqRmx0x1jowxXWtJZU6vsr1nlUf8Uvw0oq+t6tYv5EFjfV75rZlLMlEvGSuzE2C5w6ZWc+x4OiW37th1du3/L3Ir67hqIi8WfzD0+es1J/10thU5+/OwnGb6lOZ/zXKkK/olptSQqecnIwF2qI7hviDE2CjtZrytuhx/wq8rMDYkOBGrl9kBSLeJbu/SXqJbpwidtKJ1QpJwF4XgML28gJSWLIU4Rxg7njK2XhAyYI6P28oHRVtzSqwErOseKXxqbTDamAYWTuwdNrYcR+VTkNPySBvcdvgbN/w07ZXaqtJLjnacBjdzbAKk2+ZX5hp51b53zroxfDeapZ4uG2Y5oY/fJ648IZOvsr2Gs9MqLz1zKovYOsNwfIgx15q5ojaMJGZn/xqyEEErTff3g1dorFhW4p2Nz2B8RT+BClBsA8JjDP00gUe+3icahPjo3ViDR0G5c/I0nTwJg2s6B/7voFa5R1zKl2O4GiLczWFxGLGuo7mgboZqoohGvNjOdJkG4+gVJoiOa74VuhR32SR2RH8Vno5gf/7i0nga/QZQSwMEFAAAAAgAAAAhXAAAAAACAAAAAAAAABQAAABwaXBlbGluZS9fX2luaXRfXy5weQMAUEsDBBQAAAAIAAAAIVyAxI7/fRMAABZBAAAZAAAAcGlwZWxpbmUvcmVnZW5lcmF0ZV9zMS5web07a3PjNpLf/SuwmKtaykNz7CSb2miWl5pNPLep7N5Ozczd1ZWiY9EUJGNNkVyC9COO//t1NwASICHLyVWdPtgiATQa3Y1+i3P+XV0VfduKqmPfC9F8EuKGtSJXdSWrHduJSrR5J+vqLatqllfqTrRnRQ0Liq4SSrGtLDvRwtyEc35yIvdN3XYsVw9VIWv7WNTNw/Bd3dqvbV5t6v3wJOy3Tu7Fybat92yTdwKfmBmxzzHN+bmuzDyxz2WZ9J0slZ3a5K0SOD/r6syu07ObvLsu5ZWd+QEeB8yvu665P9Hzinq/rys7DY7jvE1kbQfyphHVJvsH0KyM2UbuhOpiJOLw7q6VgAc+uN/Lk5OTjdjSUVSX75tosTxh8GlF17fVcNikqu8ie144ZLFIpKq3dbvPu2hhgNzmpaTTqrpvCxG19Z2KAddqK3cGrGpEwVLzbsX1RL6mMbllpaho1YL9LqW5Ky7u4V8nNhm+52tWt+Z404kAswMRyvQoX+sN6Sy5VIL9Z1724rJt6zbin2hfALTdihYECGnaXQvWyKoSG7YRCCzfywpOs2Hq4kckRK5ExxcEFQ7OYHsmK/ynxq1woMpRWKrhkEVd9vsKcE9uEQVlSWw/cO6qBt4qWQELKk24ZCe6CCEtYqa6doHnxlkwtMLX6wTeymYKK3jcLf+bVArvkqY368R9t2SPCOeJW+6VNQiL4ZxGHSS8vhGV2cJeBiTDIOG0yLw8OcphMxGG3XWR5l8rmpqvYyJfOrAUgOBL1ZSyM2/pO7ycHdylgbgFetZVamHrRwRFZ0r1ybSoAwcRJQkyZVDSA8/Ks3tLcMBQEd60D1m+BYWUKRSjjYqI7YaKhtn0ypFQDec8OadXAGI2ts/vIxiP2RZo1xmYGgtxX4imczg+LvYghaAFVZSBnjhagZ2RjqBXkdnX2TsaN4/Z54fGfv37rWgB4Tt6XARRwUOfnBRlDpr8PZC/fPfhBy23H/sKt3PXgoJ/18NNhYGCbELMrmRZomzDBZEVsQwg/xPuWccErlRs38N31dUNnBZG7+r2RlsK4pg2KJnWZRlcFLyi+mXM6r5r+s4Mjjh8dy2KG9a0QgEmhAcDXfpABgpOIves3jLXRAFyucWbKVFurQGDpYRLWBFYNFwNoN/Nr78h54rv9WXPtqC+ygxwbOpKDUqWzgfivtLP+7y9ES08tyIRqsgbEXlnXnENxlBJT+drI3jVBjYaLg+AgMkbsC1t1PLoW7lf/M+K/dStT6Nvlz+d/nS6+Jaz13bL1wznmPd6Go/N6YyAGYtgtiFdfzEeWB8l0aYvGo5dt9mmBxVRoFCHkDe4l8q5gIb0euJqaXZcnaOazVswcQf07QEcUMQEatOyRw7zxYGNnH1w/WL567bRpwMlU5S9Gvd5xd6X+Y7lViS6moxbXW7APvXtmdJyB2Zg35RAJRLb0eG6Etf5rcTrpAw4vE+yBUsIkOoCfDWAvZX38KLq91fAy9p4EDEJOhyvLNkdSDvrwZ4WcAHy9iGxLAUxUSJvi+u5kLx6vIi/fqLH17+QZAxDq4uzr9arZLHWb0CU5vqfBOpDmVd4jl8uUYvR5fzlo9iW+tC/XN43Za39SQK+dKRTg/6XUQwPChudHii/q2Qnb0VmSJpdg8cFe3PPPOi1g5L7KEaXFlSc3gM1UZZJAJdlESqImLnmZtBEbV13Dlo4MzEGDD3cZAMONH7R3pE/T4OazXQNGn6mZm/YI3bBjPM9fQHQnUkr7g0aHWTkIELb7ytYvOGPXHUPpQAmBFXP04ztIHITlaUBrBEaBxeub0kPGvmey41RrI7efYkK1Fr5ELCXQLB3fWoXZ+7qpfGB2ewwzFD9TnbXdClhmzO9DfPUnaH5S9DSxKh8Pjbg/TWeVx1G9f0cA21/r9AnBj21gWMAcPK2NcwRwbyRU/mBV77UwAswcfVGlCOmjxyFWUHceHb71RmAReEZXm3h1l3zp2OY/0el+gadWkBwCEP1RosZAgMDMrEFYnYuLuDu4P7XcneN/8HVesHm8JyzEYHuWlY3yF8DfmZAXCcBESpsDF08oGjKqiOHYT7G/uTa0DA2zoKBeTlrakXKDoGDCmuPIwVHz9AhlkLNkPLGAKnzY0g5CxykqhoszC5/Hq8L9qd03JVcf9o0ZV9+8+UXF1/bS3uefPOHYSr4i1mjZx0l2A/G8RzkhvZghZDWNdXQQoJErmqG5hMup40Z9L7nI7VkldmpeMwW9nPnHiffDxPfGDYcDZVDq/OBADqUQZMk/c28caToMPZyJGDpHIWiBo9/qgHUHmjJ15gJgEikRHF5Ru4IxChqA/c1aPgypBheiivtzxTtzZT8WfCJWR11gY70WeorqdeMh6wPrTWq9/BCPYF7clM9ROGNdToCzx0AbgYnqYujZPg37auATjJ5DHDuMK/0AKEfOP1vgZfbnpIL6BxCuEfJJZtp2EpRgmBMKEYkzTBUKCgW0Um45CP9i8LcB1W+gbgj0YyIJlhXOxENnF3Emt3jrq/Y3yE8YyZFFLNKAKKsrIu8pEwcuDZXoqwxljG2CcKmfp9fAcvlBqO2zvivxAPEYD9EfqlNSK0wixdl4JaXIssWCaXfMNcSAUY0JuskOLx2rqHZbgT7aPMgS/digGXx8YBh/8XTVE7BdQSohIjrTbI3jLc9aET44snfmwGZ1fLi6/UcWrK/2cg2anLM3ar0c9uLWItFVt/Q44jCPq/kFhUPeHge87jdBA9gvsLhftuZ/aQQ9wTNwvJeThZU4i6z+bolWwWvWRy4XZNsFEcdXYGrk3Vwo2gthJE70ZnpFhUPxgREm99lwI7MugKTpTOs/NVO8iHT+XFY+D6HoHcyEbfAu5UNzsd84uhzj5njaJSpN+jkaO4mOARej32eiOA/QCuAf2hVvF2ddxiKdopWl3yqXuEsG1ITQzo7cqFhnrvJZQskluVE6mgexrha8MaDkBokyIOrazbyFaIDYKUnrIC5G3HP12tCCV/522Vwmcg3TzUdT4YAT/VFAaQWGxPhEaB5CsfZk9LANC0G7Bf0COENhAGgVFmaIqPxAnSCO9vkt8LGkISfs4VbKJhRkeYGqW8D3+Cco+Sh6VSP0XHuXQ5hrirrjjBwsaM5FMwYUmIcCnr6xudKjhBshSdRJbhd0ZDS1CsrUKzWZbLJy30N/gBcmgJirsWczf6adLYGDPI8NlnNIsAjztqMGmaq4VhRSqE9GF82jsZGV/XmYa5ctS5feood1KhVKjDyyLuHRsAXLio0eBv+NNVE02BneSgIQsjk6C49H3oCz3HCl3O3HGBAXCzy/QGFtcd00o7U+Zz2j7ytSzqMelCgVLi2JGj1fVMyRrRP80LCCKVXoDjnMNDNWBGHAvwf5GlWAFpx4jQVIqb7rkPqdkvpNFKO5LCRnxMIn16zi0XohmixHq/bROgxtQmO3VzSJ3l6UpOYmdFXe8mMRrLRC1iPoJwdqNBwDRFW5UhRp8oQD7YALk4G3hmBNASAQz5NEMMABLbKUfBt8WSgHdYSBh3sjsyKIkSwUfVYvWJisuj5QC1QgdPI6bw7OvXEC32xE4iiQYW/Ka7z7o3R3SAPCoQMdXKKt3ieVBrZsOJYHc6MCdA6Vm+U6HfgDmzEsxgl5ORjDsiAicIbYi3MhU9Wfz5V45X0DeYMo2d3zuQmRajajMkNX8TjGAmNM2xSL2Eh0rcbE+AQ8DctaFln5XwQN+pRaziz6DkEf37E4roGXxE9clgN95ke8T6fr2dzjXrCLCtNW1mFxedzBwVqE86wyszWOLpOnlZAcBBvxvB+BhyOL9X1gIeerl+acDSwyHoTsCzMSgs1RS1dNxwQ31CUPcEbYsK8DGyAH1ziVrtHKlCdC4ev6rocB4ZE6UvAeTWzAZZfNZsDCtDPLQjiLQ7VCQ8gZDB4JhH+G++RIUg6k5zUbupxONVP4TtkmJSGORde450+9Z4OXFTSMCncdp1OsQTARIZPYSzIOe5smNdyO4qoXiCrYclLrjLZDIrrUyvLlLQF8em3W1mgjs6MAgHVpLsX0Cxd6djIt0Cm+k0NO8lfPn/+8ImOS6kTMCM4PrcOmiSwP4wmL9LeLtJmNSL91fkfY/bV+Tfw54tvnijtowf/NWV/OD+fS7SxiNRHYkHOZr1i39c0pax3Q7YQ61qixXJUKyhGB6Oqi33tlYQYt30AaW8xqUI1d3RGwTNJjgi5EQ6KQWkdUJr+p1uO1GSPesZTQBp84x9qu/DIaw5ghRxmn9FsHtAFIGSWRymSNmzgXxqreKwPhRkYuMwH4lD4gVMPAp9QJX42izs5tm3j0JL8GRxNhckA29ChfR7diHFQrl0pxSj8NzMf4xHk3iLJMuwGyrIwsm7PyY/iwXz7AX3TeSfKuw5U/1Xfif+vU/C/5SVqOLH5aIRwid0Pxw5n3TxgYaMwLTPk3UHK6x5i8JlcnFlPPmZfBsJaSgdMA3f8oApGlXCohczvxtnyobqh76bB9PeOQ/r79dNb8DewOcZTFIFafeDDr/ISTThZbXs/lOg67JGIMS0LIZBo+yph3xmdv9HdW3jETcJnxxuZiSBtIJGmgRLUISK4TUjRSIIOrwgaizOt8GxlStxf5z1GRG8HQ2VwfBENeN4KcxoQZGxe65ifHTPEASpgwp3S1CJwck/HB6W8byvv9VVe3NTbLSokWUWHKjzPaxV2yr5gp6cWZR+rV+wvILR9yz6S+n2H6peJW2AqtYpAiIQXW2wUw0niHu6MFqAzi5qpp/lm5VAuyCyKXZ0IWm+SfTGtGGLMyFFWGE+Z3QhQoqcx1ql3QGib4O6sdkz/HTB0wj/O+Ufih070KSXUW1Pq2uaytIJQVyxnmDOjhrVa97El7M94uXWHmlN/HZrDDF+9DGOoduPL6+drqRjKihzrNnmB1dJAT5I+8rGC0PeXlx8+XV7+mIFmyH68/G8EbRuEXKBDdYd6w8yjtykWbJQYx6j9A1/aF+RgVA+RxAInNfiha6OXmSzMgnIkEv0hu+qlhT3sJLaNYrZeZUsFkwynk9L1bEKXqxunmU5fI4ziUDLGWdN8g+43/udmn+R9N/Rw44sTX7SP5fwcBK3zgZGKuQt/hcdpyifkgEwzJ41uS4MRauWTMxJbidHmZUhqy4lToUmBx4q6GkxJ6vIW7xppjxRuTDSDNGfrwX7fjVBFOmrmsbkKTHEPe6S2csyxM16hQkwRe3utfaQdt047Q+/wxXeUvJn7Xle5ElnflilpRfuEatI4nOkjNazWrfzZVqy2/M8C1HzLHs2Ne5pmXElotM+VPp98crXR8G0OrJR72alUH+iv9IA6ErMKle5cU2mgjeRFVhvh3ADx4U7diiMQJ0wkL0znxQ4l5FBHUzdlG2p1xw8JCvp8KCxGdA974G5e1Cbe/ZT7waUgztbru3h+Fii1DPN8W3kfWS/g/yzmsw+E6iC+1yklOBcncwmyysnqA3DJqNEb3keWphqDbMwuB7cfPIJJm5F7oReLeYIrmGnFj2+0dzn4dm10ShgHM1l5WR6AhOjjOirc4/rDvMfhpEAPszyQ8nwGrdj4TJmOPkjA/VpjEE3QlKgGwUaiykGzMMcPRaYoayWCyvpgNRE9pLabOi6YNvV9kst7/ROhsmRDvt40l5Kaq3oY0ekpReREZ6UH0SJXU8HNbslC6pAYuxfU1CvBTccGON2zgf2bAPkljXc3VX1XmfMQrF/nR9DulJGkjXVC0Cz7XRoo+h/D6BO1/Fh8sNeshxnYFqK7nnW/iukJ4mFc6OxTVHSb/NCtMnoxx3tA3yOXXJTo1wftrUAXGtMv2oExrT3mCassbj+cbkKYlN/3bspJz/g1jR1aLsTEDcr7HTZlUHXJee0p6zA3bLXpcBXcm+9krKfada7OB6xsPftxdhlPT6cVvrlGDjZgLIfQfWz2Ws8TlqgCDkD0ukFGcKYF7EWwJsSxvBmOa36PlU0LeaaeZDf1+gxg2JjVQ1nh4TOpL/jgJrWHY6B0NsUHod8dx8JNLE+x8MZiEM6j4Jzy1QSYO/JyMLZWGoRkyl4OI1v8ZakxAehEBD3vuSMxXtB+v8cELRZwBwMA2/ve+Jh590fY2YhA8IR8GKYj2anMETXTfRVsubJQDjcsjSl3OLXug8e7HrAFvam8rVqvxIeR3uOTbvmZdvtQlmjoWTGjtMl6SkD9E0XMs5lehUw3tKMKGEvx0w4MzPEtiW/98JvOmJ0TUueEU4+7Eqr+5XV/S2rbFWyXxMgv/NXJ+JLirKGVYgAXbOAyxuANcEof0DZvmcdFYFHpriL1YHq24kHXPGealxPVXappfDv0SsyISAYGl7hdFDHTbyC+2pmfuw26dzZmFemBi/ryZj8/WgYvEcPcgS7ImLyVACYp1K2f9sO0WN3q24gLE/TAMio93Ucc5yfdvpmk88hLG1YmNWjjiONvDMANr1H8Ut5327M/nim549jcelfKSqScU3y1nTucxE+MzXG/72XR/Re9iLaxbtglcqf0F80+BJYqp5gu5XJX1W2oOqdhJvRPx74BH9udhBY2GgzyIkwm4EdT5oWIkFjOr069H++FuBBx81PeBHiBaQYjpzorbBWqLiHamXrGhPaO6OMeMQugbPr2HjnOALWlJ9prxZf2Rg2GNjMTD1ymp5P/BVBLAwQUAAAACAAAACFcpmClS7wFAAA0CwAAIwAAAGNvbmZpZ3MvcmVnZW5lcmF0ZV9zMV9kZWVwc2Vlay55YW1sbVbfj+M0EH7fv2IELyB2u/2xba/hCYk7CYE4xJ10Ogmpcp1J4qtjB9tptwL+d75x0m0W8dTU9sx88803Y0ffB83FHVHgzhf0gXtXf+ndY9ntA7f+xOU+Ln5+WMwW+czJRONdQZtysd0eqs1cLXbrUq/08vC0Knmj3nBVLZZP87Va6N3TEkbau8rUBZVcqd4mrMTOmlRQCso4/OXnjnVCoODPsaDdbjNYJXZpX5qaIw5vtuunZbXmarWsdrytVtv1XK0OO4Ti9dN8sy23rLfz+dNis1T8plwpvZwvd5vdcqeqzWp9p73tWxcl1T97uMxpXL+w6IOpjVMWaavonXEZMneR+bhPjXFHLO2B+QvA+nCZmigXzxwm51VK3HbpTnVGAh5U5H0fbEFNSl0sHh+xMbuenmnf4lDrS7Y3Hw+np4cu+Mz6CGgPbn0AGY2pG2wk3+27ghazOf58Te87DkqyUZY0GwsLwvmbPX1HA9J7+vX9R1LYyQRwSUmFmhNZdnVqZtndD+O/B2taI2cCx867yGQiVVbVNZf35PjEAaVkJUdUhFek01lOjMIq+RJ3rXreJ39kFIAWq8V8OypD9yGw0xesStVb4/YjqL2BAMIJ7EbGwRKG89ky0zHsJ9Oy79Nte/FmPh9DBU7BMNZW2SCFC9yZZKbeli9bYvGyvJnfxRZIi6zMnIAcFs8oS1kQRI2wHSKD3FYlORjTxXJBTqU+TCWErWqiEEQKRxHKV+9kdaxG8dUdCg25FPT3Ayw+eHtiSg1T7Du0CnjF/sFyS8bRW1dbE5sZfUbrDt5vlTkba+nA1EepqafESjekBGGrrEWlssqo8WfZHnAiVPB93VyjRLLmKABMnNFHwMCxEqbZuXf2ggXo68QiwUbE8AoENOETXQRd5E5BktlX7p/HmxZ1o5xje0/R51yv5nBqUmRbUdvHlOcAxkQ+UvXW0qv2DOaUFR/v6cgX0srq3l5XlCvh7Au8mMroYXVGP/qMD+POKp1pbpFZAkugu0WBLuSrSc+gqaTO4JNt5HPDAYKG30/BJH6FfFD/qIGph+A1xzjwoqjzUj+pDz+ng/dHMGB7QSeN8pPTti8Hx5W31p/FxYEbdTI+IEDKW51He0TKcGThco2L6niNrioE4wP9ZsFyJiuixRMdLmRKzFVTXcSvuKq9QhXyF2rqqFIankGecB/zkE4Dm1lIInapaX0Z1jC9rRTo3MB1ghZgnFUxqmmWcbxFI/dqGLrnxlgZDqyR9QgCPtw9NMH6SLnxQbgR5YBcXBqRMGobsFdhK4PItR+tW/om9geUOfWwOCh9vLqKphYl9Oh9/ETcM0aGI8Yx/mI0gj/YhzGxVtU4KOwLfKv6aJDBt0MGv3NlMfnHDMCTGmNUytgozhTWOmQtLQe2rZWQ6iICF5KMNsmCtCv/OeS5QUHPWKBz8K4eKM3X7CCBq8ORRbjxQU1B/E/MIa+8Z1zJHTuJSC1S9SWdfW9RyVLCKJtB5BsaZzR6F/MwO7UgGpKCIkYptGhllgrnZhFsrZT9BhCuPr9q+lsHtGABAuZrN/MQHNShpQZlAjeHIArH5ECFMI+SpAKv1xA87KkDAHn4mNEnSROf0rFANFxNRkTba4EOssd2UvmyEVZxXcXxxq8knWlDjf2E1dxDQcZ9rpDElRsFF56MTvmLZjZauliMB6lfR871SGWeccCk2UtX5+k55im0lpishPpIyyAMOq+/tXAQY5cn6fQV8DLCKnUIMtekQhfEqIahe6M96wqXmU+NCK8czKT0YiAFGz3huhjBxkHi1AAYXGBe/sJpmiMgS3eihoM+W98jiK+Gdxy6OKvze9kMuRzOT54ZZx9K0Wd+Cry8NmD7FmRkYf3nZswsjUMWc0CKC6XSHwcPtH/9Mwh9Cq+RIQyHpYk6yCOkmvib0TvpUzmM4uNzAvnlLXgvghxHOw7iBTw+VWwf85D+F1BLAQIUAxQAAAAIAAAAIVwAAAAAAgAAAAAAAAASAAAAAAAAAAAAAACAAQAAAABjb21tb24vX19pbml0X18ucHlQSwECFAMUAAAACAAAACFcdC3HJdwCAACGCAAADAAAAAAAAAAAAAAAgAEyAAAAY29tbW9uL2lvLnB5UEsBAhQDFAAAAAgAAAAhXAAAAAACAAAAAAAAABQAAAAAAAAAAAAAAIABOAMAAHBpcGVsaW5lL19faW5pdF9fLnB5UEsBAhQDFAAAAAgAAAAhXIDEjv99EwAAFkEAABkAAAAAAAAAAAAAAIABbAMAAHBpcGVsaW5lL3JlZ2VuZXJhdGVfczEucHlQSwECFAMUAAAACAAAACFcpmClS7wFAAA0CwAAIwAAAAAAAAAAAAAAgAEgFwAAY29uZmlncy9yZWdlbmVyYXRlX3MxX2RlZXBzZWVrLnlhbWxQSwUGAAAAAAUABQBUAQAAHR0AAAAA'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE))) as bundle:
    bundle.extractall(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT))
print("Project code ready:", CODE_ROOT)

## Settings and generation prompt

Defaults come from the bundled `configs/regenerate_s1_deepseek.yaml`, shown below.
Edit `CONFIG` in this cell before loading the source if needed. The default is
**`deepseek-v4-pro`, high thinking effort, 16 concurrent requests** with request pacing
and bounded retries for rate limits and transient API failures. Temperature is omitted
because DeepSeek ignores it in thinking mode. The sampling seed fixes the 20 selected
examples; hosted generation itself is not guaranteed deterministic.

`max_tokens=131072` is an operational ceiling for **reasoning + answer**, not a requested
length or an SFT limit. It is not included in the prompt. A response that hits this ceiling
is flagged as incomplete; its raw text is saved in the journal, but is not placed in the
completed-output columns. No source rows are removed by the old 20,480-token SFT limit.

Changing the config or bundled generation code creates a separate run folder. Reuse the
same notebook/settings to resume. Hosted model aliases may change server-side: returned
model IDs, timestamps, fingerprints, usage, and response IDs are recorded per request.

In [ ]:
import yaml
CONFIG = yaml.safe_load((CODE_ROOT / "configs/regenerate_s1_deepseek.yaml").read_text())
# Optional edits go here, before creating the run in the next cell.
# CONFIG["api"]["concurrency"] = ...
print(yaml.safe_dump(CONFIG, sort_keys=False, allow_unicode=True))

In [ ]:
# @title Load the pinned Hugging Face source and prepare a resumable run
import importlib.metadata
from pipeline.regenerate_s1 import RegenerationRun, load_source, timestamp
from common.io import append_jsonl, read_jsonl
source_rows = load_source(CONFIG, HF_TOKEN)
run = RegenerationRun(source_rows, CONFIG, PROJECT_ROOT)
runtime_path = run.root / "runtime_sessions.jsonl"
read_jsonl(runtime_path, repair_tail=True)
append_jsonl(runtime_path, {"started_at": timestamp(), "python": sys.version,
    "packages": {name: importlib.metadata.version(name) for name in
                 ["datasets", "huggingface-hub", "httpx", "PyYAML", "tqdm"]}})
print("Source rows:", len(source_rows))
print("New columns:", run.reasoning_column, run.answer_column)
print("Smoke sample indices (seeded):", run.smoke_indices)
print("Drive output folder:", run.root)

## Smoke test — 20 random examples and a comparison CSV

This cell makes paid API requests for the 20 sampled rows that are not already complete.
The CSV has exactly these five columns, in this order:

`question`, `deepseek_thinking_trajectory`, `deepseek_attempt`,
`deepseek-v4-pro_reasoning`, `deepseek-v4-pro_answer`.

UTF-8 CSV preserves multiline text, quotation marks, and Unicode. Long reasoning can exceed
spreadsheet apps' cell-display limits; the CSV itself retains the full text. Blank generated
cells mean generation is incomplete/failed/pending or the final-response format is invalid;
inspect `smoke/status.jsonl` for details. Format checks require text before a single
`Final answer:` line and a nonempty conclusion, and flag headings from the previous fixed
cognitive-section template. Proofs can end with a textual conclusion; no box or length
minimum is enforced. These checks do not assess correctness or quality. Review whether
intermediate checks occur where useful and compare any reported corrections with the raw
API reasoning column. A textual match alone does not prove the model's internal history.
A smooth solution without corrections is valid. Raw returned text,
including format failures, remains in `attempts.jsonl`. Rerun to retry unsuccessful rows.
The old `deepseek_grade` is not a grade for the new answer, and no grading is run here.

In [ ]:
# @title Smoke test: generate the fixed random sample and save/download the CSV
try:
    await run.generate(run.smoke_indices, DEEPSEEK_API_KEY)
finally:
    smoke_result = run.export(run.smoke_indices, "smoke")
    print(smoke_result["summary"])
    print("CSV saved:", smoke_result["path"])
    print("Row status:", smoke_result["status_path"])
files.download(str(smoke_result["path"]))

## Full run — all 996 rows, preserving original columns

Set `RUN_FULL_GENERATION = True` below to run. There is no baseline-evaluation or smoke-completion gate.
Matching completed smoke results are reused automatically. After a runtime restart, run
setup/settings/source-loading, then this cell directly to resume.

The export is a Hugging Face-compatible **JSONL dataset** in original source order, with
every original field plus the two generated fields. Existing grades and responses retain
their original meaning; they are never overwritten. The source HF repository is unchanged.

`full/dataset.jsonl` is written when every row is complete. Otherwise the export is called
`full/dataset.partial.jsonl`, still containing every row, with null generated fields for
unsuccessful rows (including missing final markers or fixed cognitive headings). `full/status.jsonl`
lists failures by source index; `attempts.jsonl`
retains raw returned text, including incomplete responses and retry usage. Rerun the cell
to retry remaining rows. Authentication/balance/invalid-request errors stop the run and
save partial results. Failed responses are not silently promoted into the training data.

This notebook only regenerates data. It does not run SFT, evaluation, or HF publication.

In [ ]:
# @title Full run: generate all rows (completed smoke rows are reused)
RUN_FULL_GENERATION = False
if RUN_FULL_GENERATION:
    full_indices = list(range(len(source_rows)))
    try:
        await run.generate(full_indices, DEEPSEEK_API_KEY)
    finally:
        full_result = run.export(full_indices, "full")
        print(full_result["summary"])
        print("Dataset saved:", full_result["path"])
        print("Row status:", full_result["status_path"])
    if full_result["summary"]["remaining"]:
        print("Some rows remain incomplete. Inspect status.jsonl and rerun this cell.")
    else:
        print("All source rows now have generated reasoning and answers.")
else:
    print("Full run is off. Enable RUN_FULL_GENERATION to process all source rows.")

Load the finished export later with
`load_dataset("json", data_files=".../full/dataset.jsonl", split="train")`.
Drive stores the dataset and journals; save a notebook copy in Drive to retain cell output
displays too. The API journal records returned token usage, including retries. A request
whose response was lost or cancelled can still be billed; consult the provider dashboard.

References: [Kimi k1.5, §2.2](https://arxiv.org/html/2501.12599v1#S2.S2),
[DeepSeek thinking fields](https://api-docs.deepseek.com/guides/thinking_mode/),
[DeepSeek API](https://api-docs.deepseek.com/api/create-chat-completion/).